# HW2: generate model programs on one NVIDIA T4

Run these cells in order. The model was trained from random weights on the allowed training split. Each submitted program is an exact decoded model output; there are no handwritten repairs. The timer includes tokenization, generation, sorting and CSV writing. Provenance packaging is saved separately.

The original test.parquet order is preserved. Direct comparison with the separate pass1_questions.jsonl remains pending; equal row counts alone do not establish alignment.

In [ ]:
!git clone --depth=1 --branch=2026 https://github.com/OsVox/hse-nlp.git /content/hse-nlp
%cd /content/hse-nlp
!pip -q install 'transformers>=4.53,<5' 'pyarrow>=18' 'requests>=2.31'
!git rev-parse HEAD

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
assert 'T4' in gpu, 'Final speed verification requires an actual NVIDIA T4'

In [ ]:
from pathlib import Path
from homeworks.hw2.inference import generate_predictions
count, seconds = generate_predictions(
    model_dir=Path('homeworks/hw2/model'),
    output_file=Path('/content/predictions.csv'),
    cache_dir=Path('/content/hw2_cache'),
    batch_size=64, max_new_tokens=192, precision='fp32',
)
assert count == 1919
assert seconds <= 180, f'Generation took {seconds:.2f}s'
print(f'{count} model programs in {seconds:.2f}s on {gpu}')

In [ ]:
import csv, json, hashlib
from transformers import PreTrainedTokenizerFast
with open('/content/predictions.csv', newline='', encoding='utf-8') as source:
    programs = [r['code'] for r in csv.DictReader(source)]
provenance = json.loads(Path('/content/predictions.provenance.json').read_text())
tokenizer = PreTrainedTokenizerFast.from_pretrained('homeworks/hw2/model')
assert programs == tokenizer.batch_decode(provenance['selected_token_ids'], skip_special_tokens=True, clean_up_tokenization_spaces=False)
assert len(programs) == 1919 and all(p.strip() for p in programs)
output = Path('/content/predictions.jsonl')
output.write_text(''.join(json.dumps({'code':p},ensure_ascii=False)+'\n' for p in programs), encoding='utf-8')
assert [json.loads(line)['code'] for line in output.read_text().splitlines()] == programs
provenance['jsonl_sha256'] = hashlib.sha256(output.read_bytes()).hexdigest()
Path('/content/predictions.provenance.json').write_text(json.dumps(provenance))
print('JSONL SHA256:',provenance['jsonl_sha256'])
print('Model SHA256:',provenance['model_sha256'])

In [ ]:
from zipfile import ZipFile
from google.colab import files
with ZipFile('/content/hw2_inference_result.zip','w') as archive:
    for name in ['predictions.csv','predictions.jsonl','predictions.provenance.json']:
        archive.write('/content/'+name,name)
files.download('/content/hw2_inference_result.zip')

Upload only predictions.jsonl to the course bot. Keep the provenance file for the report. Stop and delete the Colab runtime after downloading the results. A measured T4 runtime below 180 seconds establishes speed for that checkpoint and configuration only.